In [2]:
##checking normality of features
# if majority are normal will proceed with normal standardization, otherwise will rank features

from scipy.stats import shapiro
import numpy as np
import pickle
import pandas as pd


#importing data

try:
    with open('frameVectorAnnots.pkl', 'rb') as f:
        annots = pickle.load(f)
except:
    print("Error")

In [9]:
#stacking all frames into matrix
frameList = [] 

frameIds = [] 

for day in annots:
    for operation in annots[day]:
        for frame in annots[day][operation]:
            frameList.append(frame['frameVector'])
            frameIds.append((day, operation, frame['frameIdx']))


frameMatrix = np.array(frameList)
print(np.shape(frameMatrix))

#------converting matrix into pandas data frame--

#creating cols and rows
dfCols = []
dfRows = []

interpersonIdx = 1
tableFeatIdx = 1
for i in range(1, np.shape(frameMatrix)[1]+1):
    if i <=50:
        dfCols.append(f"Gpd_{i}")

    elif i == 51:
        dfCols.append(f"Num_clinicians")
    
    elif i>=52 and i<=55:
        dfCols.append(f"Table_feature_{tableFeatIdx}")
        tableFeatIdx+=1
    
    else: 
        dfCols.append(f"Interperson_feature_{interpersonIdx}")
        interpersonIdx+=1

for i in range(1, np.shape(frameMatrix)[0]+1):
    dfRows.append(f"Frame_{i}")

(626, 64)


In [10]:
print(dfCols)

['Gpd_1', 'Gpd_2', 'Gpd_3', 'Gpd_4', 'Gpd_5', 'Gpd_6', 'Gpd_7', 'Gpd_8', 'Gpd_9', 'Gpd_10', 'Gpd_11', 'Gpd_12', 'Gpd_13', 'Gpd_14', 'Gpd_15', 'Gpd_16', 'Gpd_17', 'Gpd_18', 'Gpd_19', 'Gpd_20', 'Gpd_21', 'Gpd_22', 'Gpd_23', 'Gpd_24', 'Gpd_25', 'Gpd_26', 'Gpd_27', 'Gpd_28', 'Gpd_29', 'Gpd_30', 'Gpd_31', 'Gpd_32', 'Gpd_33', 'Gpd_34', 'Gpd_35', 'Gpd_36', 'Gpd_37', 'Gpd_38', 'Gpd_39', 'Gpd_40', 'Gpd_41', 'Gpd_42', 'Gpd_43', 'Gpd_44', 'Gpd_45', 'Gpd_46', 'Gpd_47', 'Gpd_48', 'Gpd_49', 'Gpd_50', 'Num_clinicians', 'Table_feature_1', 'Table_feature_2', 'Table_feature_3', 'Table_feature_4', 'Interperson_feature_1', 'Interperson_feature_2', 'Interperson_feature_3', 'Interperson_feature_4', 'Interperson_feature_5', 'Interperson_feature_6', 'Interperson_feature_7', 'Interperson_feature_8', 'Interperson_feature_9']


In [11]:
frameVectorDf = pd.DataFrame(
    frameMatrix,
    columns = dfCols,
    index = dfRows
)

frameVectorDf.head()

,Gpd_1,Gpd_2,Gpd_3,Gpd_4,Gpd_5,Gpd_6,Gpd_7,Gpd_8,Gpd_9,Gpd_10,...,Table_feature_4,Interperson_feature_1,Interperson_feature_2,Interperson_feature_3,Interperson_feature_4,Interperson_feature_5,Interperson_feature_6,Interperson_feature_7,Interperson_feature_8,Interperson_feature_9
Frame_1,9.522396,9.745949,-2.441483,-2.424845,0.847117,-3.612987,-7.905068,-4.840728,7.682698,2.057512,...,1767.585850,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_2,-6.240740,7.566539,4.392485,-3.616573,8.635847,-11.345625,-5.310072,4.286849,0.439230,3.040247,...,2622.616756,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_3,-7.432102,-9.433077,-2.599083,-2.519187,6.199976,1.432799,-7.219241,7.671942,0.163997,3.052592,...,2445.534774,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_4,-10.756118,-3.976860,-1.695456,-4.020653,6.824022,-1.232486,-3.858355,-1.802732,6.092485,-1.278292,...,2456.068758,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Frame_5,-9.633286,-4.889347,0.764947,-1.182092,8.913971,3.765080,-4.183628,3.302828,1.759320,2.955471,...,2400.928740,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# computes shapiro wilk for each in feature in the data frame
    # Ho: data is normally distributed vs Ha: data is non normally distributed
# returns a dataframe showing which features are normally distributed and which arent
def computeShapiro(df, alpha = 0.05):
    results = []
    rows  = [] 
    cols = ["Normality"] 
    for colName, colData in df.items():
        rows.append(colName)

        #checking normality
        _, p = shapiro(colData)
        normality = 0 if p<=alpha else 1   #1 if data is normal, 0 if not
        results.append(normality)

    featureDistrib = pd.DataFrame(
        results,
        columns = cols,
        index = rows
    )

    #compute percentage of normally distributed features
    normalFeatures = len(featureDistrib.query('Normality == 1'))

    return featureDistrib, normalFeatures

In [23]:
distrib, numNormal = computeShapiro(frameVectorDf)

print(f"{numNormal}/64 features are normally distributed")
distrib.head(64)


8/64 features are normally distributed


,Normality
Gpd_1,0
Gpd_2,1
Gpd_3,0
Gpd_4,1
Gpd_5,0
...,...
Interperson_feature_5,0
Interperson_feature_6,0
Interperson_feature_7,0
Interperson_feature_8,0


In [24]:
print(distrib.query('Normality == 1'))

        Normality
Gpd_2           1
Gpd_4           1
Gpd_11          1
Gpd_12          1
Gpd_15          1
Gpd_22          1
Gpd_24          1
Gpd_44          1


In [31]:
distrib, numNormal = computeShapiro(frameVectorDf, 0.01)

print(f"{numNormal}/64 features are normally distributed")

10/64 features are normally distributed
